In [ ]:
# =================================
# IMPORT des Bibliothèques
# =================================
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm

# ==============================================
# --------------- METHODE 1 --------------------
# ==============================================

In [ ]:
# ==============================
# OPEN AGENDA
# ==============================

# Récupération des agendas
headers = {'key':'oa_pk_IttDjfMFnvJZfuumozNLWwTGAorNMAlHxNItNOAVEJgJwyKsYwMBZIgWxhpxDsjn'}

liste_data_temp = []
after = 0
for i in tqdm(range(1,10000,100)):
    url = f"https://api.openagenda.com/v2/agendas?size=100&official=1&after[]=1&after[]={after}"
    response = requests.get(url, headers=headers)
    api_response = response.json()
    data_temp = pd.json_normalize(api_response, record_path='agendas')
    liste_data_temp.append(data_temp)
    after = api_response['after'][1]

# Création du df_agenda
df_Agenda = pd.concat(liste_data_temp)
df_Agenda = df_Agenda.drop_duplicates(subset=['uid'])

In [ ]:
# Récupération des événements
list_id_evenement = df_Agenda['uid'].to_list()
liste_data_temp = []

for uid in tqdm(list_id_evenement):
    after = 0
    url = f"https://api.openagenda.com/v2/agendas/{uid}/events?relative[]=current&relative[]=upcoming"
    response = requests.get(url, headers=headers)
    api_response = response.json()
    data_temp = pd.json_normalize(api_response, record_path='events')
    liste_data_temp.append(data_temp)

# Création du df evenements
df_OA = pd.concat(liste_data_temp)

In [ ]:
# outil de test valeurs manquantes colonne -----  cible < 5% + mots clé
colonne = []
for col in df_OA.columns:
    # ratio = pourcentage de valeurs manquantes
    ratio = df_OA[col].isna().sum() / len(df_OA)
    x = re.search(r"keywords.fr", col)
    if ratio < 0.05:
        colonne.append(col)
    elif x != None:
        colonne.append(col)

df_OA = df_OA[colonne]
df_OA = df_OA.drop_duplicates(subset=['uid'])
df_OA.info()

# ==============================================
# --------------- METHODE 2 --------------------
# ==============================================

In [ ]:
# ===============================
#############" TEST TEST TEST "
# ===============================
liste_data_test = []

# Nombre d'enregistrement à récupérer
url_record = f'https://userclub.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records'
response_record = requests.get(url_record, headers=headers)
api_response_record = response_record.json()
temp = pd.json_normalize(api_response_record)
record = (temp['total_count'][0])//100

for i in tqdm(range(0,10000,100)):
    url = f'https://userclub.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records?limit=100&offset={i}&lang=fr&timezone=Europe%2FParis&include_links=true&include_app_metas=true'
    response = requests.get(url, headers=headers)
    api_response = response.json()
    data_temp = pd.json_normalize(api_response, record_path='results')
    liste_data_test.append(data_temp)

df_test = pd.concat(liste_data_test)

In [ ]:
# outil de test valeurs manquantes colonne -----  cible < 5% + mots clé
colonne = []
for col in df_test.columns:
    # ratio = pourcentage de valeurs manquantes
    ratio = df_test[col].isna().sum() / len(df_OA)
    x = re.search(r"^age|keywords.fr|category", col)
    if ratio < 0.05:
        colonne.append(col)
    elif x != None:
        colonne.append(col)
print(colonne)
print(len(colonne))

df_test = df_test[colonne]
df_test.info()